# 14b · Rebuild `14_detector_scores.npz` for Fig. 3 (CPU, ~2–5 min)

Notebook 14's last cell, which saves the per-image detector scores, was never run. This notebook rebuilds the file from what notebook 14 did save:
- the cached ImageNet DINO features: `features_v2/imagenet_dino/<target>_{train,val,test}.npz`;
- the perceptual hashes: `results_v2/14_hashes/<target>_{train,val,test}.npz`.

No images and no GPU are needed. For every test image it computes the nearest train/val cosine similarity and the nearest pHash Hamming distance. It then **checks the flagged sets against `14_drop_sets.json`** (the sets the paper used) before saving.

Send back **`results_v2/14_detector_scores.npz`** and the printed check table.

In [ ]:
import os, sys, json
import numpy as np
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive; drive.mount('/content/drive')
BASE = os.environ.get('SSL_BASE', '/content/drive/MyDrive/SSL_AgriStudy')
R, FEATS = f'{BASE}/results_v2', f'{BASE}/features_v2'
HDIR = f'{R}/14_hashes'
TARGETS = ['FloraBD', 'RiceLeaf', 'PlantVillage', 'PlantDoc', 'Flowers102', 'CUB']
drop = json.load(open(f'{R}/14_drop_sets.json'))
for t in TARGETS:
    for s in ('train', 'val', 'test'):
        for f in (f'{FEATS}/imagenet_dino/{t}_{s}.npz', f'{HDIR}/{t}_{s}.npz'):
            assert os.path.exists(f), f'missing {f}'
print('inputs present')

_LUT = np.array([bin(i).count('1') for i in range(256)], dtype=np.uint8)
def popcount64(x):                                   # identical to notebook 14
    if hasattr(np, 'bitwise_count'): return np.bitwise_count(x)
    return _LUT[x.view(np.uint8)].reshape(*x.shape, 8).sum(-1)
def nearest_hamming(q, r, chunk=256):
    best = np.empty(len(q), np.int16)
    for i in range(0, len(q), chunk):
        best[i:i + chunk] = popcount64(q[i:i + chunk, None] ^ r[None, :]).min(1)
    return best
def nearest_cosine(Q, Rf, chunk=1024):
    Q = Q / np.linalg.norm(Q, axis=1, keepdims=True); Rf = Rf / np.linalg.norm(Rf, axis=1, keepdims=True)
    return np.concatenate([(Q[i:i + chunk] @ Rf.T).max(1) for i in range(0, len(Q), chunk)])

out, rows = {}, []
for t in TARGETS:
    F = {s: np.load(f'{FEATS}/imagenet_dino/{t}_{s}.npz') for s in ('train', 'val', 'test')}
    H = {s: np.load(f'{HDIR}/{t}_{s}.npz') for s in ('train', 'val', 'test')}
    for s in F:
        assert len(F[s]['y']) == len(H[s]['ph']), f'{t}/{s}: {len(F[s]["y"])} features vs {len(H[s]["ph"])} hashes'
    cb = nearest_cosine(F['test']['X'].astype(np.float32),
                        np.concatenate([F['train']['X'], F['val']['X']]).astype(np.float32))
    hb = nearest_hamming(H['test']['ph'].astype(np.uint64),
                         np.concatenate([H['train']['ph'], H['val']['ph']]).astype(np.uint64))
    exact = set(np.where(np.isin(H['test']['md5'], np.concatenate([H['train']['md5'], H['val']['md5']])))[0].tolist())
    out[f'{t}_cos'], out[f'{t}_ham'] = cb.astype(np.float32), hb
    u = lambda idx: exact | set(np.asarray(idx).tolist())
    chk = {'phash8': u(np.where(hb <= 8)[0]), 'phash4': u(np.where(hb <= 4)[0]), 'phash12': u(np.where(hb <= 12)[0]),
           'cos0.90': u(np.where(cb >= 0.90)[0]), 'cos0.98': u(np.where(cb >= 0.98)[0]), 'cos0.95': u(np.where(cb >= 0.95)[0])}
    r = {'target': t, 'n_test': len(cb)}
    for k, v in chk.items():
        ref = set(drop[t][k]); r[k] = 'same' if v == ref else f'{len(v)} vs {len(ref)} (sym. diff {len(v ^ ref)})'
    rows.append(r)

import pandas as pd
print('Recomputed flagged sets vs 14_drop_sets.json ("same" = identical set of test images).')
print('cos0.95 in the paper is notebook 11\'s GPU list, so a few borderline differences there are expected:')
display(pd.DataFrame(rows).set_index('target'))
np.savez(f'{R}/14_detector_scores.npz', **out)
print('saved', f'{R}/14_detector_scores.npz')

In [ ]:
if IN_COLAB:
    from google.colab import drive; drive.flush_and_unmount(); print('✓ uploaded to Drive')